# GGA Network Training - Step 4: Refactored Library-Driven Training

This notebook reproduces the Step 3b experiment (H / O / H2O at def2-svp,
12 architectures x 6 loss approaches = 72 models) using the refactored
`xcquinox.alec` subpackage. All loss, network, training, and evaluation
logic lives in the library -- this notebook is a thin orchestration layer
that builds `Spec` objects and calls `run_pretrain`, `run_training`, and
`run_test`.

## Training Methodology

| Approach | Energy Calculation | Density Matching | Description |
|----------|-------------------|------------------|-------------|
| **A** | Fixed-density | None | AE only on PBE density |
| **B** | Fixed-density | One-shot DM -> HF target | AE + DM correction learning |
| **C** | Fixed-density | One-shot grid rho -> HF target | AE + grid density correction |
| **D1** | Fixed-density | None | Delta-learning energy only |
| **D2** | Fixed-density | One-shot DM -> HF target | Delta-E + DM correction |
| **D3** | Fixed-density | One-shot grid rho -> HF target | Delta-E + grid density correction |

## Key Change from Step 3b

Step 3b inlined the loss, network, training loop, and evaluation code inside
the notebook. Step 4 delegates every step to `xcquinox.alec`:

- `alec.PretrainSpec` / `alec.run_pretrain` -- pretraining phase
- `alec.TrainingSpec.from_dicts` / `alec.run_training` -- main training phase
- `alec.TestSpec.from_dicts` / `alec.run_test` -- evaluation phase

The registry-driven composition means adding a new loss or architecture is a
single-line library change, not a notebook edit.

## Network Architectures (12 total)

**Standard (2 inputs: rho, sigma):**
`shallow`, `shallow_attn`, `medium`, `medium_attn`, `deep`, `deep_attn`

**Extended features (deep only):**

| Architecture | Inputs | Dimension |
|--------------|--------|-----------|
| `deep_cusp`, `deep_cusp_attn` | $[\rho, \sigma, f_{cusp}, \log Z]$ | 4 |
| `deep_dm`, `deep_dm_attn` | $[\rho, \sigma, f_{idem}, f_{entropy}, f_{offdiag}]$ | 5 |
| `deep_combined`, `deep_combined_attn` | $[\rho, \sigma, f_{idem}, f_{entropy}, f_{offdiag}, f_{cusp}, \log Z]$ | 7 |

**Total: 72 models** = 12 architectures x 6 training approaches


In [ ]:
import os
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
# JAX config: pin x64 dtype and CPU device *before* importing jnp or any
# library that may trigger JAX tracing. These must not change later in the
# notebook -- flipping jax_enable_x64 after traces are cached produces
# inconsistent dtypes.
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_device", jax.devices("cpu")[0])
import jax.numpy as jnp
import equinox as eqx

from pyscf import gto, dft, scf, cc

import xcquinox.alec as alec
import xcquinox.features


In [ ]:
BASIS = 'def2-svp'
CHECKPOINT_BASE = 'checkpoints_step4'
GRID_LEVEL = 1
PRETRAIN_ATOMS = (("H", 1), ("He", 0), ("O", 2), ("N", 3))
H2O_COORDS = "O 0.0000 0.0000 0.1173; H 0.0000 0.7572 -0.4692; H 0.0000 -0.7572 -0.4692"

os.makedirs(CHECKPOINT_BASE, exist_ok=True)
print(f"CHECKPOINT_BASE={CHECKPOINT_BASE}  BASIS={BASIS}  GRID_LEVEL={GRID_LEVEL}")


In [ ]:
# Print all 12 registered architectures from alec.ARCHITECTURES.
# Fields printed: name, depth, nodes (hidden size), attention flag, descriptors.
_header = f"{'arch_name':<22} {'depth':>6} {'nodes':>6} {'attention':>10}  descriptors"
print(_header)
print("-" * len(_header))
for _name in alec.ARCHITECTURES.keys():
    _cfg = alec.get_architecture(_name)
    _descs = ", ".join(s.name for s in _cfg.descriptors) or "-"
    print(f"{_name:<22} {_cfg.depth:>6} {_cfg.nodes:>6} {str(_cfg.attention):>10}  {_descs}")


In [ ]:
ARCH_NAMES = list(alec.ARCHITECTURES.keys())

cmap = plt.get_cmap("tab20")
arch_colors = {name: cmap(i / max(1, len(ARCH_NAMES) - 1)) for i, name in enumerate(ARCH_NAMES)}

print(f"Selected {len(ARCH_NAMES)} architectures:")
for _n in ARCH_NAMES:
    print(f"  {_n}")
